<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c39-header.png" alt="Nextia Learning · Entity Resolution and Record Matching: Is This the Same Thing?" width="100%">

# Blocking keys and sorted neighbourhood

**[Blocking keys and sorted neighbourhood](https://learning.nextia-ai.com/courses/record-matching/m03/blocking-keys-and-sorted-neighbourhood/)** · Entity Resolution and Record Matching: Is This the Same Thing? · Module 3, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build blocking keys, measure each one, combine them, and write a sorted-neighbourhood blocking.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (about 6 MB). |
| **You should know** | Candidate pairs and pair completeness, from [Why all pairs is impossible](https://learning.nextia-ai.com/courses/record-matching/m03/why-all-pairs-is-impossible/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (rapidfuzz 3.14.6, jellyfish 1.3.0, datasketch 2.0.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/record-matching/m03/blocking-keys-and-sorted-neighbourhood/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/M03-L02-blocking-keys-and-sorted-neighbourhood/blocking-keys-and-sorted-neighbourhood-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 8 MB) into a folder `larkfield-match` and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-match` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Every customer record and every product listing is **synthetic**: our code made them from invented people and products, and then added realistic errors on purpose (typing errors, nicknames, old addresses, shared family emails). Because the code knows the true person behind each record, you can always check an answer. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/data/dataset-card.md) explains the error model.

In [ ]:
# Setup: download the files into larkfield-match/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C39/"
FILES = {
    "data/sources/shop_accounts.csv": "f59f3a9619c35b572650e10663a0ba5f316202d173850689ec9242706fb4174e",
    "data/sources/loyalty_members.csv": "629696f38469898be48cb664e5d69f9c6d438e8622899bbb281bd520bf44af32",
    "data/sources/garden_centres.csv": "2878063d24a38d841f93bb14e07b2d46619fcc4a3909e92ce0238116434d85b2",
    "data/truth.csv": "d951378129bdf30b649250e6d2127787e5630d66aa7cfe5793b1a110e37a06d3",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-match").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-match and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-match.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd` and matplotlib. It defines a few small helpers:

- `source(name)` reads one customer source as it is (`"shop"`, `"loyalty"` or `"garden"`); `records()` reads all three in shared columns (`record_id, source, name, email, phone, address, postcode, date_of_birth`), with the values exactly as the sources wrote them.
- `truth()` says which true customer (`entity_id`) each record belongs to, and its split (`train`, `validation` or `test`).
- `pair_measures(pairs)` measures a table of pairs (`record_id_l`, `record_id_r`) that a matcher calls "the same": pair precision, recall and F1 on the test split, as in the project's `match eval`.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, pair measures, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

FILES_OF = {"shop": "shop_accounts.csv", "loyalty": "loyalty_members.csv", "garden": "garden_centres.csv"}

def source(name):
    """One customer source as it is: its own columns, every value as text."""
    return pd.read_csv(f"data/sources/{FILES_OF[name]}", dtype=str, keep_default_na=False)

def records():
    """All customer records in shared columns, values as the sources wrote them."""
    s, l, g = source("shop"), source("loyalty"), source("garden")
    parts = [
        pd.DataFrame({"record_id": s.record_id, "source": "shop", "name": s.first_name + " " + s.last_name,
                      "email": s.email, "phone": s.phone, "address": s.address + ", " + s.city + " " + s.province,
                      "postcode": s.postcode, "date_of_birth": s.date_of_birth}),
        pd.DataFrame({"record_id": l.record_id, "source": "loyalty", "name": l.full_name, "email": l.email,
                      "phone": l.phone, "address": "", "postcode": l.postcode, "date_of_birth": l.date_of_birth}),
        pd.DataFrame({"record_id": g.record_id, "source": "garden", "name": g.name, "email": g.email,
                      "phone": g.phone, "address": g.address, "postcode": "", "date_of_birth": g.date_of_birth}),
    ]
    return pd.concat(parts, ignore_index=True)

def truth():
    """record_id -> entity_id (the true customer) and split."""
    return pd.read_csv("data/truth.csv", dtype=str)

def ordered(pairs):
    """The same pairs with the smaller record ID first, one row per pair."""
    lo = pairs[["record_id_l", "record_id_r"]].min(axis=1)
    hi = pairs[["record_id_l", "record_id_r"]].max(axis=1)
    out = pd.DataFrame({"record_id_l": lo, "record_id_r": hi})
    return out[out.record_id_l != out.record_id_r].drop_duplicates()

def true_pairs():
    """Every pair of records of the same true customer, with its split."""
    t = truth()
    m = t.merge(t, on="entity_id", suffixes=("_l", "_r"))
    m = m[m.record_id_l < m.record_id_r]
    return m[["record_id_l", "record_id_r", "split_l"]].rename(columns={"split_l": "split"})

RANK = {"train": 0, "validation": 1, "test": 2}

def pair_measures(pairs, split="test"):
    """Pair precision, recall and F1 of the pairs a matcher calls the same (a pair is in the test split when one
    of its records belongs to a test customer)."""
    p = ordered(pairs)
    t = truth().set_index("record_id")
    rank = t.split.map(RANK)
    p = p.assign(split=np.maximum(p.record_id_l.map(rank), p.record_id_r.map(rank)).map({v: k for k, v in RANK.items()}),
                 same=(p.record_id_l.map(t.entity_id) == p.record_id_r.map(t.entity_id)))
    tp_all = true_pairs()
    p, tp_all = p[p.split == split], tp_all[tp_all.split == split]
    tp = int(p.same.sum())
    precision = tp / len(p) if len(p) else 0.0
    recall = tp / len(tp_all) if len(tp_all) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3),
            "pairs": len(p), "right": tp, "true_pairs": len(tp_all)}

def same_value_pairs(df, col):
    """Every pair of records that share a non-empty value of col."""
    d = df[df[col].notna() & (df[col] != "")][["record_id", col]]
    sizes = d[col].value_counts()
    n_pairs = int((sizes * (sizes - 1) // 2).sum())
    if n_pairs > 20_000_000:   # protect the computer's memory from a key that is too wide
        raise ValueError(f"This key makes {n_pairs:,} pairs: too many to hold in memory. Use a narrower key.")
    m = d.merge(d, on=col, suffixes=("_l", "_r"))
    return m[m.record_id_l < m.record_id_r][["record_id_l", "record_id_r"]].drop_duplicates()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## Setup: the clean records of Module 2

The next cell cleans every record as Module 2 did: names without accents and titles, split into first name and
surname; emails in lower case; phones as 10 digits; postcodes as 6 characters; dates as YYYY-MM-DD. Run it. It
prints the number of records.

In [ ]:
# The clean records (Module 2): one row per record with first, last, email, phone, postcode, dob.
import re, unicodedata

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def norm_text(s):
    s = strip_accents(str(s or "")).lower().replace("'", "")
    return " ".join(w for w in re.sub(r"[^\w\s]", " ", s).split() if w not in {"mr", "mrs", "ms", "dr", "m", "mme"})

def norm_phone(s):
    d = re.sub(r"\D", "", re.split(r"(?i)\s*ext", str(s or ""))[0])
    d = d[1:] if len(d) == 11 and d.startswith("1") else d
    return d if len(d) == 10 else ""

def norm_postcode(s):
    p = re.sub(r"[\s-]", "", str(s or "")).upper()
    return p if re.fullmatch(r"[A-Z]\d[A-Z]\d[A-Z]\d", p) else ""

def norm_date(s):
    s = str(s or "")
    m = re.fullmatch(r"(\d{1,2})/(\d{1,2})/(\d{4})", s)
    return f"{m[3]}-{int(m[2]):02d}-{int(m[1]):02d}" if m else (s if re.fullmatch(r"\d{4}-\d\d-\d\d", s) else "")

def split_name(s):
    s = str(s or "")
    if "," in s:
        last, first = s.split(",", 1)
    else:
        words = s.split()
        first, last = " ".join(words[:-1]), (words[-1] if words else "")
    return norm_text(first), norm_text(last)

def postcode_of(postcode, address):
    """The postcode column, or the postcode at the end of a garden-centre address."""
    found = re.findall(r"[A-Za-z]\d[A-Za-z][ -]?\d[A-Za-z]\d", address)
    return norm_postcode(postcode) or (norm_postcode(found[-1]) if found else "")

recs = records()
names = recs.name.map(split_name)
clean = pd.DataFrame({
    "record_id": recs.record_id, "source": recs.source,
    "first": [f for f, _ in names], "last": [l.replace(" ", "") for _, l in names],
    "email": recs.email.str.strip().str.lower(), "phone": recs.phone.map(norm_phone),
    "postcode": [postcode_of(p, a) for p, a in zip(recs.postcode, recs.address)],
    "dob": recs.date_of_birth.map(norm_date),
})
print(f"{len(clean):,} clean records")

## 1. A function that measures a blocking

Every blocking gives a table of candidate pairs. Run the next cell: `measure` computes the **pair completeness** (the share of true pairs among the candidates) and the **reduction ratio** (the share of all pairs removed).

In [ ]:
tp = true_pairs()
all_pairs = len(clean) * (len(clean) - 1) // 2

def measure(cand):
    """Candidate pairs, pair completeness and reduction ratio of a table of candidate pairs."""
    kept = tp.merge(cand, on=["record_id_l", "record_id_r"])
    return {"candidates": len(cand), "pair_completeness": round(len(kept) / len(tp), 3),
            "reduction_ratio": round(1 - len(cand) / all_pairs, 6)}

print(f"{len(tp):,} true pairs, {all_pairs:,} possible pairs")

## 2. Five keys, one at a time

> **Predict.** Which of the five keys keeps the most true pairs: postcode, email, phone, surname or date of birth?

The next cell builds the five keys of the project from the clean values, and measures each one.

In [ ]:
first1, first2 = clean["first"].str[:1], clean["first"].str[:2]
keys = pd.DataFrame({
    "record_id": clean.record_id,
    "postcode": clean.postcode,
    "email": clean.email,
    "phone": clean.phone,
    "surname": np.where(clean["last"] != "", clean["last"].str[:4] + "|" + first2, ""),
    "dob": np.where(clean.dob != "", clean.dob + "|" + first1, ""),
})
KEY_NAMES = ["postcode", "email", "phone", "surname", "dob"]
pairs_of = {k: same_value_pairs(keys, k) for k in KEY_NAMES}
results = pd.DataFrame({k: measure(p) for k, p in pairs_of.items()}).T
print(results)
results.pair_completeness.plot.bar(title="Pair completeness of each key", ylim=(0, 1))
plt.show()

## 3. Wide keys make huge blocks

A block of *k* records makes *k* × (*k* − 1) / 2 pairs. The next cell counts the pairs that three surname keys would make, without building them.

In [ ]:
def pairs_if_blocked(values):
    """The pairs a key would make, counted from its block sizes (nothing is built)."""
    sizes = values[values != ""].value_counts()
    return int((sizes * (sizes - 1) // 2).sum())

for name, values in [("first letter of the surname", clean["last"].str[:1]),
                     ("first 4 letters of the surname", clean["last"].str[:4]),
                     ("4 letters of the surname + 2 of the first name", keys.surname)]:
    print(f"{name:<48} {pairs_if_blocked(values):>13,} pairs")

## 4. Your turn: the five keys together

A pair is a candidate when it shares **any** key. Make the union of the five tables in `pairs_of` (each pair once), and measure it.

In [ ]:
union = ...   # the candidate pairs of the five keys together: join the five tables, keep each pair once
union_result = measure(union) if union is not ... else None
print(union_result)

In [ ]:
expect_hash('union_result', union_result, '793008574e32a046')

> **Check.** Together, the five keys make 1,108,588 candidate pairs and keep 0.994 of the true pairs: much more than any key alone.

## 5. Your turn: sorted neighbourhood

Sort the records by name and pair each record with the next `window - 1` records. In `sorted_pairs`, the pairs that are `k` places apart are `ids[:-k]` with `ids[k:]`. Write that line.

In [ ]:
def sorted_pairs(df, col, window):
    """Sort by col and pair each record with the next window - 1 records."""
    d = df[df[col] != ""].sort_values([col, "record_id"])
    ids = d.record_id.to_numpy()
    parts = []
    for k in range(1, window):
        a, b = ...   # the record IDs k places apart: ids without its last k, and ids without its first k
        parts.append(pd.DataFrame({"record_id_l": np.minimum(a, b), "record_id_r": np.maximum(a, b)}))
    return pd.concat(parts).drop_duplicates()

named = clean.assign(full=(clean["first"] + " " + clean["last"]).str.strip())
sn_result = None
try:
    sn_result = measure(sorted_pairs(named, "full", 10))
except TypeError:
    print("Not yet: write the line a, b = ... in sorted_pairs.")
print(sn_result)

In [ ]:
expect_hash('sn_result', sn_result, '6b441dc3f94b19f7')

> **Check.** With a window of 10 on the full name, pair completeness is only 0.543. The full name starts with the first name, so every "Samuel" with another surname sorts between two records of Samuel Green.

Run the next cell: does sorting by the surname first help?

In [ ]:
if sn_result is not None:
    by_surname = clean.assign(name=(clean["last"] + " " + clean["first"]).str.strip())
    print("sorted by full name (first name first), window 10:", sn_result)
    print("sorted by surname first, window 10:              ", measure(sorted_pairs(by_surname, "name", 10)))

## 6. In the project

```sh
match block --list
match block --blocking surname
match block --blocking keys
match block --blocking sorted-name
```

The project's clean values are a little different from this notebook's (its parser also reads the street), so its numbers differ a little too.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [MinHash and LSH](https://learning.nextia-ai.com/courses/record-matching/m03/minhash-and-lsh/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/record-matching/m03/blocking-keys-and-sorted-neighbourhood/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every customer record and every product listing here is synthetic: our code made them from invented people and products. No real person is in this data.